##Reuters GPT-2 model

In [ ]:
!pip install transformers torch evaluate accelerate
!pip install -U datasets

In [ ]:
from datasets import load_dataset
dataset = load_dataset("HFS26/custom_reuters_articles")
dataset

In [ ]:
def create_full_article_col(example):

  return {'full_article': f"TITLE:{example['title']}\n\nBODY:{example['body']}"}

dataset = dataset.map(create_full_article_col)
dataset

In [ ]:
dataset['train'][0]['full_article']

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("HFS26/gpt2-reuters-tokenizer")

In [ ]:
CONTEXT_LENGTH_PARAM = 256

def tokenize(element):
    outputs = tokenizer(
        element["full_article"],
        truncation=True,
        max_length=CONTEXT_LENGTH_PARAM,
        return_overflowing_tokens=False
    )

    return outputs

tokenized_datasets = dataset.map(
    tokenize, batched=True, remove_columns=dataset["train"].column_names
)
tokenized_datasets

###Training model

In [ ]:
from transformers import AutoTokenizer, GPT2LMHeadModel, AutoConfig

config = AutoConfig.from_pretrained(
    "gpt2",
    vocab_size=len(tokenizer),
    n_ctx=CONTEXT_LENGTH_PARAM,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

print(f"config : {config}")

In [ ]:
model = GPT2LMHeadModel(config)
model_size = sum(t.numel() for t in model.parameters())
print(f"GPT-2 model size : {model_size/1000**2:.1f} M parameters")

In [ ]:
from transformers import DataCollatorForLanguageModeling
tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

In [ ]:
# login hugging face hub
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./reuters-gpt2-text-gen",  # local directory
    hub_model_id="HFS26/reuters-gpt2-text-gen",  # identifier on the Hub
    eval_strategy="epoch",
    auto_find_batch_size=True,
    num_train_epochs=2,
    gradient_accumulation_steps=8,
    weight_decay=0.1,
    lr_scheduler_type="cosine",
    learning_rate=5e-4,
    fp16=True,
    push_to_hub=True,
    logging_steps=10
)

trainer = Trainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
)

In [ ]:
trainer.train()

###saving model

In [ ]:
trainer.push_to_hub()

In [ ]:
# using model in Pipeline
import torch
from transformers import pipeline

pipe = pipeline(
    "text-generation", model="HFS26/reuters-gpt2-text-gen"
)

In [ ]:
sample = dataset['test'][2]

prompt = f"TITLE:{sample['title']}\n\nBODY:"
pipe(prompt, max_new_tokens=128)

prompt = f"TITLE:{sample['title']}"
pipe(prompt, max_new_tokens=128)

### continue training pre-trained model

In [ ]:
# Load model
from transformers import AutoTokenizer, AutoModelForCausalLM
pre_tokenizer = AutoTokenizer.from_pretrained("HFS26/reuters-gpt2-text-gen")
pre_model = AutoModelForCausalLM.from_pretrained("HFS26/reuters-gpt2-text-gen")

In [ ]:
training_args = TrainingArguments(
    output_dir="./reuters-gpt2-text-gen",  # local directory
    hub_model_id="HFS26/reuters-gpt2-text-gen",  # identifier on the Hub
    eval_strategy="epoch",
    auto_find_batch_size=True,
    num_train_epochs=1,
    gradient_accumulation_steps=8,
    weight_decay=0.1,
    lr_scheduler_type="cosine",
    learning_rate=5e-4,
    fp16=True,
    push_to_hub=True,
    logging_steps=10
)

trainer = Trainer(
    model=pre_model,
    #tokenizer=pre_tokenizer,
    processing_class=tokenizer,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
)

In [ ]:
trainer.train()

In [ ]:
#saving model
trainer.push_to_hub()